<a href="https://colab.research.google.com/github/saadishaq2/slms_fine_tuning/blob/main/notebooks/colab_controller.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab Controller for SLMs Fine-Tuning

This notebook is the main entry point for running the fine-tuning project on Google Colab. It sets up the environment, clones the repository, installs dependencies, and runs the training scripts.

## 1. Clone Repository & Set Working Directory

This cell clones the project repository from GitHub and changes the current working directory to the project root.

In [1]:
from google.colab import userdata

# Load the token securely
try:
    git_token = userdata.get('GIT_TOKEN')
    print("GitHub token loaded successfully.")
except:
    print("Error: GIT_TOKEN not found in Secrets.")
    git_token = ""

# Clone using the token for authentication
# The {git_token} syntax allows Python variables to be used in shell commands
if git_token:
    !git clone https://{git_token}@github.com/saadishaq2/slms_fine_tuning.git
else:
    print("Cannot clone: No token provided.")

%cd slms_fine_tuning

## 2. Install Dependencies

Install the required Python packages. We use `unsloth` for memory-efficient fine-tuning and install other dependencies from `requirements.txt`.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -r requirements.txt

## 3. Set Environment Variables & Secrets

This cell configures the output directory as an environment variable and loads secrets (like API keys) using Colab's secret manager, as recommended in the `README.md`.

In [ ]:
import os
from google.colab import userdata

# Set the output directory for model artifacts in the Colab environment
output_dir = "/content/slms_fine_tuning/outputs"
os.environ['OUTPUT_DIR'] = output_dir
print(f"Output directory has been set to: {output_dir}")

# Ensure the directory exists
os.makedirs(output_dir, exist_ok=True)

# Load secrets from Colab's secret manager
try:
    os.environ['MLFLOW_TRACKING_USERNAME'] = userdata.get('MLFLOW_TRACKING_USERNAME')
    os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('MLFLOW_TRACKING_PASSWORD')
    os.environ['HUGGING_FACE_TOKEN'] = userdata.get('HUGGING_FACE_TOKEN')
    os.environ['MLFLOW_TRACKING_URI'] = userdata.get('MLFLOW_TRACKING_URI')
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('LANGFUSE_PUBLIC_KEY')
    os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('LANGFUSE_SECRET_KEY')
    os.environ['LANGFUSE_HOST'] = userdata.get('LANGFUSE_HOST')
    # Note: MLFLOW_TRACKING_URI is typically loaded from .env locally, but can be set here or as a Colab secret if preferred.
    # Example: os.environ['MLFLOW_TRACKING_URI'] = "https://dagshub.com/<YOUR_DAGSHUB_USERNAME>/<YOUR_REPOSITORY_NAME>.mlflow"
    print("Secrets loaded successfully.")
except Exception as e:
    print("Could not load secrets. Make sure they are configured in Colab's Secrets Manager.")
    print(f"Error: {e}")

## 4. Run Training

Execute the training script. The `OUTPUT_DIR` environment variable will override the path specified in `configs/config.yaml`.

In [ ]:
!python scripts/train.py

## 5. Prepare RAG Knowledge Base

In [ ]:
!python scripts/prepare_rag_data.py

## 6. Run RAG Evaluation

In [ ]:
!python scripts/evaluate.py

## 7. Deploy Interface

This will generate a public Gradio link.

In [ ]:
!python deployment/app.py